# 08 - Training, Validation, Early Stopping & Checkpoints

## Learning Objectives
- Understand train/validation/test splitting and why each matters
- Build a complete training loop with validation monitoring
- Implement early stopping and model checkpointing
- Build a minimal neural network from scratch in NumPy, then transition to PyTorch

## Prerequisites
- Notebooks 01-07 (all DL100 fundamentals)
- Basic PyTorch (tensors, autograd)

## Table of Contents
1. [Train / Validation / Test Split](#1)
2. [Training Loop Anatomy](#2)
3. [Monitoring & Loss Curves](#3)
4. [Early Stopping](#4)
5. [Model Checkpointing](#5)
6. [Minimal NN from Scratch (NumPy)](#6)
7. [Same Network in PyTorch](#7)
8. [Exercises](#8)
9. [Common Mistakes & Debugging Tips](#9)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset, random_split
from sklearn.datasets import make_classification

np.random.seed(42)
torch.manual_seed(42)

<a id='1'></a>
## 1. Train / Validation / Test Split

- **Training set** (~70-80%): model learns from this
- **Validation set** (~10-15%): tune hyperparameters, monitor overfitting
- **Test set** (~10-15%): final evaluation only (never train on this!)

**Key rule:** The test set must NEVER influence training decisions.

In [ ]:
# Generate synthetic binary classification data
X, y = make_classification(n_samples=1000, n_features=20, n_informative=10,
                           n_redundant=5, random_state=42)
X = torch.tensor(X, dtype=torch.float32)
y = torch.tensor(y, dtype=torch.float32)

dataset = TensorDataset(X, y)
train_set, val_set, test_set = random_split(dataset, [700, 150, 150],
                                            generator=torch.Generator().manual_seed(42))

train_loader = DataLoader(train_set, batch_size=32, shuffle=True)
val_loader = DataLoader(val_set, batch_size=64)
test_loader = DataLoader(test_set, batch_size=64)

print(f"Train: {len(train_set)}, Val: {len(val_set)}, Test: {len(test_set)}")

<a id='2'></a>
## 2. Training Loop Anatomy

```
for epoch in range(num_epochs):
    model.train()               # 1. Set train mode
    for X_batch, y_batch in train_loader:
        optimizer.zero_grad()   # 2. Clear gradients
        output = model(X_batch) # 3. Forward pass
        loss = loss_fn(output, y_batch)  # 4. Compute loss
        loss.backward()         # 5. Backward pass
        optimizer.step()        # 6. Update weights
    
    model.eval()                # 7. Set eval mode
    with torch.no_grad():       # 8. No gradients for validation
        val_loss = evaluate(...)  
```

In [ ]:
# Simple MLP for binary classification
class SimpleMLP(nn.Module):
    def __init__(self, input_dim, hidden_dim=64):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, 32),
            nn.ReLU(),
            nn.Linear(32, 1)
        )
    def forward(self, x):
        return self.net(x).squeeze(-1)

model = SimpleMLP(20)
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
loss_fn = nn.BCEWithLogitsLoss()

<a id='3'></a>
## 3. Monitoring & Loss Curves

In [ ]:
# Complete training loop with monitoring
train_losses, val_losses = [], []
num_epochs = 50

for epoch in range(1, num_epochs + 1):
    # --- Train ---
    model.train()
    epoch_loss = 0
    for X_b, y_b in train_loader:
        optimizer.zero_grad()
        loss = loss_fn(model(X_b), y_b)
        loss.backward()
        optimizer.step()
        epoch_loss += loss.item()
    train_losses.append(epoch_loss / len(train_loader))
    
    # --- Validate ---
    model.eval()
    val_loss = 0
    with torch.no_grad():
        for X_b, y_b in val_loader:
            val_loss += loss_fn(model(X_b), y_b).item()
    val_losses.append(val_loss / len(val_loader))
    
    if epoch % 10 == 0:
        print(f"Epoch {epoch}/{num_epochs} | Train: {train_losses[-1]:.4f} | Val: {val_losses[-1]:.4f}")

# Plot
plt.figure(figsize=(8, 4))
plt.plot(train_losses, label='Train Loss')
plt.plot(val_losses, label='Val Loss')
plt.xlabel('Epoch'); plt.ylabel('Loss')
plt.title('Training & Validation Loss')
plt.legend(); plt.grid(True, alpha=0.3)
plt.tight_layout(); plt.show()

<a id='4'></a>
## 4. Early Stopping

Stop training when validation loss stops improving.

- **Patience**: number of epochs to wait after last improvement
- Save the best model weights when validation loss improves
- Restore best weights when stopping

In [ ]:
import sys; sys.path.insert(0, "../..")
from src.utils.training import fit, EarlyStopping
from src.utils.seed import set_global_seed

set_global_seed(42)

model2 = SimpleMLP(20)
optimizer2 = torch.optim.Adam(model2.parameters(), lr=1e-3)
early_stop = EarlyStopping(patience=7)

history = fit(
    model2, train_loader, val_loader,
    epochs=100, optimizer=optimizer2,
    loss_fn=nn.BCEWithLogitsLoss(),
    device=torch.device('cpu'),
    early_stopping=early_stop,
    verbose=True
)

In [ ]:
from src.utils.plotting import plot_loss_curves
plot_loss_curves(history, title='Training with Early Stopping')

<a id='5'></a>
## 5. Model Checkpointing

In [ ]:
import os, tempfile

# Save checkpoint
checkpoint_dir = tempfile.mkdtemp()
checkpoint = {
    'epoch': len(history['train_loss']),
    'model_state_dict': model2.state_dict(),
    'optimizer_state_dict': optimizer2.state_dict(),
    'train_loss': history['train_loss'][-1],
    'val_loss': history['val_loss'][-1],
}
path = os.path.join(checkpoint_dir, 'best_model.pt')
torch.save(checkpoint, path)
print(f"Saved checkpoint to {path}")

# Load checkpoint
loaded = torch.load(path, weights_only=False)
model_loaded = SimpleMLP(20)
model_loaded.load_state_dict(loaded['model_state_dict'])
print(f"Loaded model from epoch {loaded['epoch']}, val_loss={loaded['val_loss']:.4f}")

<a id='6'></a>
## 6. Minimal NN from Scratch (NumPy)

A tiny 2-layer network showing the full forward → loss → backward → update cycle.

In [ ]:
np.random.seed(42)

# XOR problem
X_np = np.array([[0,0],[0,1],[1,0],[1,1]], dtype=np.float64)
y_np = np.array([[0],[1],[1],[0]], dtype=np.float64)

# Initialize weights (He initialization)
W1 = np.random.randn(2, 8) * np.sqrt(2/2)
b1 = np.zeros((1, 8))
W2 = np.random.randn(8, 1) * np.sqrt(2/8)
b2 = np.zeros((1, 1))

lr = 0.1
losses = []

def sigmoid(z):
    return 1 / (1 + np.exp(-np.clip(z, -500, 500)))

for epoch in range(2000):
    # Forward
    z1 = X_np @ W1 + b1
    a1 = np.maximum(0, z1)  # ReLU
    z2 = a1 @ W2 + b2
    a2 = sigmoid(z2)  # output
    
    # Loss (BCE)
    loss = -np.mean(y_np * np.log(a2 + 1e-8) + (1 - y_np) * np.log(1 - a2 + 1e-8))
    losses.append(loss)
    
    # Backward
    dz2 = a2 - y_np              # (4, 1)
    dW2 = a1.T @ dz2 / 4         # (8, 1)
    db2 = np.mean(dz2, axis=0, keepdims=True)
    da1 = dz2 @ W2.T             # (4, 8)
    dz1 = da1 * (z1 > 0)         # ReLU derivative
    dW1 = X_np.T @ dz1 / 4       # (2, 8)
    db1 = np.mean(dz1, axis=0, keepdims=True)
    
    # Update
    W2 -= lr * dW2
    b2 -= lr * db2
    W1 -= lr * dW1
    b1 -= lr * db1

print(f"Final loss: {losses[-1]:.4f}")
print(f"Predictions: {a2.flatten().round(2)}")
print(f"Targets:     {y_np.flatten()}")

plt.plot(losses)
plt.xlabel('Epoch'); plt.ylabel('Loss'); plt.title('NumPy NN: XOR')
plt.grid(True, alpha=0.3); plt.show()

<a id='7'></a>
## 7. Same Network in PyTorch

Notice how much simpler it is with autograd!

In [ ]:
torch.manual_seed(42)

X_t = torch.tensor(X_np, dtype=torch.float32)
y_t = torch.tensor(y_np, dtype=torch.float32)

model_xor = nn.Sequential(
    nn.Linear(2, 8), nn.ReLU(),
    nn.Linear(8, 1)
)
opt = torch.optim.SGD(model_xor.parameters(), lr=0.1)
loss_fn_xor = nn.BCEWithLogitsLoss()

for epoch in range(2000):
    opt.zero_grad()
    loss = loss_fn_xor(model_xor(X_t), y_t)
    loss.backward()
    opt.step()

with torch.no_grad():
    preds = torch.sigmoid(model_xor(X_t))
print(f"PyTorch predictions: {preds.flatten().numpy().round(2)}")

<a id='8'></a>
## 8. Exercises

**Exercise 1:** Implement early stopping from scratch (without using the utility).

**Exercise 2:** Add accuracy tracking to the training loop.

**Exercise 3:** Implement a training loop that saves the best model based on validation accuracy (not loss).

<a id='9'></a>
## 9. Common Mistakes & Debugging Tips

| Mistake | Fix |
|---------|-----|
| Forgetting `model.eval()` before validation | Dropout/BatchNorm behave differently |
| Forgetting `torch.no_grad()` for validation | Wastes memory building computation graph |
| Not restoring best weights after early stopping | Always save & load best state_dict |
| Using test set for hyperparameter tuning | Use validation set; test set is final evaluation only |
| Training too few epochs | Use early stopping instead of guessing epoch count |